# Exploratory Data Analysis Konsumsi Bahan Bakar (2000 sampai 2022)

**Info Ngopi**

| Anggota | NIM |
|---|---|
| Amadeus Eugene Dirgantara | 0706012410063 |
| Rei Putra Soemanto | 0706012410060 |
| Jason Tio | 0706012410006 |

**Dataset.** Fuel Consumption 2000 sampai 2022.

**Kandidat prediktor (x).** `engine_size` (ukuran mesin dalam L), `cylinders` (jumlah silinder), `year` (tahun model), `transmission` (jenis transmisi).

**Variabel respon (y).** `comb_l_100km` (konsumsi bahan bakar gabungan, diambil dari kolom asli `COMB (L/100 km)`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', lambda v: f'{v:.4f}')
sns.set_theme(style='whitegrid', context='notebook')
RNG = 42

## 1. Memuat Data Mentah

In [ ]:
raw = pd.read_csv('dataset/Fuel_Consumption_2000-2022.csv')
print('Bentuk:', raw.shape)
raw.head()

In [ ]:
raw.info()

## 2. Memeriksa Nilai yang Hilang

In [ ]:
missing = raw.isna().sum().to_frame('n_missing')
missing

*Tidak ditemukan nilai yang hilang di dataset.*

## 3. Memeriksa Data Duplikat

In [ ]:
n_before = len(raw)
raw = raw.drop_duplicates().reset_index(drop=True)
print(f'Menghapus {n_before - len(raw)} baris duplikat. Tersisa: {len(raw)}')

## 4. Inspeksi Kolom Transmisi

`transmission` sempat terdaftar sebagai salah satu kandidat prediktor. Sebelum membangun frame analisis, kita memeriksa nilai-nilai unik kolom `TRANSMISSION` pada data mentah untuk menentukan apakah kolom ini layak masuk ke model regresi polinomial.

In [ ]:
print('Jumlah kategori transmisi:', raw['TRANSMISSION'].nunique())
print('Nilai unik:', sorted(raw['TRANSMISSION'].unique()))
raw['TRANSMISSION'].value_counts()

*Kolom `TRANSMISSION` ternyata berisi kode kategorikal seperti `A4`, `A5`, `M5`, `M6`, `AS6`, `AV`, dan seterusnya, yang menandai tipe sekaligus jumlah gigi transmisi (otomatis, manual, CVT, dan varian lainnya). Karena nilainya bersifat kategorikal nominal dan bukan rasio, kolom `TRANSMISSION` tidak bisa digunakan dalam analisis korelasi maupun regresi polinomial dan lebih cocok untuk model ensemble. Oleh karena itu, `transmission` dikeluarkan dari himpunan kandidat, dan frame analisis hanya dibangun dari prediktor numerik kontinu `engine_size`, `cylinders`, dan `year`.*

## 5. Membangun Frame Analisis

In [ ]:
df = raw[['ENGINE SIZE', 'CYLINDERS', 'YEAR', 'COMB (L/100 km)']].rename(
    columns={
        'ENGINE SIZE': 'engine_size',
        'CYLINDERS': 'cylinders',
        'YEAR': 'year',
        'COMB (L/100 km)': 'comb_l_100km',
    }
).reset_index(drop=True)
df.head()

## 6. Statistik Deskriptif

In [ ]:
df.describe().T

*`engine_size` berkisar antara 1 sampai 8 L dengan median di sekitar 3 L, `cylinders` terkonsentrasi pada konfigurasi standar 4/6/8, dan `year` mencakup rentang 2000 sampai 2022 secara merata seperti yang seharusnya. `comb_l_100km` memiliki skewness kanan, dengan mayoritas mobil berada di rentang 8 sampai 14 L/100 km dan ekor yang lebih tipis melampaui 20 L/100 km.*

## 7. Plot Distribusi

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.ravel()
for ax, col in zip(axes, df.columns):
    ax.hist(df[col], bins=30, edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
fig.tight_layout()
plt.show()

*`engine_size` dan `comb_l_100km` keduanya memiliki skewness kanan dengan ekor atas panjang yang berasal dari kendaraan mewah dan performa besar. `cylinders` bersifat diskret dan terkonsentrasi pada 4, 6, dan 8 silinder. Itulah sebabnya histogram `cylinders` hanya menampilkan beberapa batang terpisah alih-alih sebaran yang kontinu. `year` terdistribusi hampir uniform, yang menunjukkan bahwa dataset menyampelkan setiap tahun model secara seimbang.*

## 8. Plot Pencar Setiap Prediktor terhadap Respon

Plot inilah petunjuk visual utama untuk mendeteksi ketidaklinieran. `engine_size` secara khusus diperkirakan menunjukkan kurva konkaf yang konsisten dengan menurunnya efisiensi pembakaran pada perpindahan piston yang lebih besar.

In [ ]:
predictors = ['engine_size', 'cylinders', 'year']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, p in zip(axes, predictors):
    ax.scatter(df[p], df['comb_l_100km'], alpha=0.15, s=10)
    ax.set_xlabel(p)
    ax.set_ylabel('comb_l_100km')
    ax.set_title(f'{p} vs combined fuel consumption')
fig.tight_layout()
plt.show()

*`engine_size` menunjukkan hubungan monotonik yang jelas dengan `comb_l_100km`, namun terlihat datar setelah sekitar 5 L. Bentuk tersebut adalah kelengkungan konkaf, yaitu pola hasil yang menurun seperti yang diprediksi secara fisik. Pola inilah yang menjadi motivasi visual untuk perluasan polinomial pada Tahap 2 dan 3. `cylinders` juga sangat terkait dengan `comb_l_100km`, namun bersifat diskret dan sangat terikat dengan `engine_size`. Keterkaitan tersebut akan dikuantifikasi oleh matriks korelasi di bawah. `year` tampak datar; pusat garis garis vertikal hampir tidak bergeser di sepanjang rentang, sehingga mengindikasikan korelasi yang nyaris nol dengan `comb_l_100km`.*

## 9. Matriks Korelasi Pearson

Korelasi Pearson menangkap asosiasi **linear**.

In [ ]:
pearson = df.corr(method='pearson')
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(pearson, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Pearson Correlation')
plt.show()
pearson

*Dua prediktor menunjukkan asosiasi linear yang kuat dengan `comb_l_100km`: `engine_size` (r ≈ 0.81) dan `cylinders` (r ≈ 0.77). `year` praktis tidak berkorelasi dengan `comb_l_100km` (|r| < 0.06), sehingga akan dibuang karena korelasi marginal yang lemah. Angka yang patut diwaspadai adalah r(`engine_size`, `cylinders`) ≈ 0.91. Nilai sebesar itu menunjukkan bahwa kedua prediktor tersebut hampir merupakan variabel yang sama, sehingga memasukkan keduanya ke dalam OLS akan menyebabkan multikolinieritas. Konsekuensinya adalah standard error membengkak dan estimasi koefisien menjadi tidak stabil, meskipun daya prediksinya secara bersama tetap nyata. Aturan Tahap 1 menangani hal ini dengan mempertahankan prediktor yang lebih kuat dari pasangan kolinier tersebut (`engine_size`) dan membuang yang lebih lemah (`cylinders`).*

## 10. Matriks Korelasi Spearman

Korelasi Spearman menangkap asosiasi **monotonik**. Kesenjangan yang mencolok antara Pearson dan Spearman pada suatu pasangan adalah bukti bahwa hubungannya nonlinier tetapi tetap monotonik. Itulah persis kasus yang ingin ditangani oleh suku polinomial.

In [ ]:
spearman = df.corr(method='spearman')
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(spearman, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Spearman Correlation')
plt.show()
spearman

*Korelasi Spearman mengukur asosiasi monotonik tanpa peduli pada bentuk kurva, sementara Pearson hanya mengukur asosiasi linear. Matriks ini sendiri kurang informatif jika berdiri sendiri; nilainya muncul saat dibandingkan terhadap nilai Pearson pada pasangan yang sama. Bila Spearman nyata lebih tinggi dari Pearson pada pasangan prediktor terhadap `comb_l_100km`, kesenjangan tersebut menjadi bukti langsung bahwa hubungannya monotonik tetapi tidak linear. Kasus tersebut tepat ditangani oleh perluasan polinomial.*

## 11. Kesenjangan Pearson vs Spearman terhadap Respon

Membandingkan kedua korelasi terhadap respon mengisolasi perilaku yang monotonik tetapi nonlinier.

In [ ]:
target = 'comb_l_100km'
gap = pd.DataFrame({
    'pearson_vs_y': pearson[target].drop(target),
    'spearman_vs_y': spearman[target].drop(target),
})
gap['abs_gap'] = (gap['spearman_vs_y'].abs() - gap['pearson_vs_y'].abs())
gap.sort_values('abs_gap', ascending=False)

*Kolom `abs_gap` melaporkan |Spearman| dikurangi |Pearson| terhadap `comb_l_100km` untuk setiap prediktor. Dengan n ≈ 20k, derau sampling pada korelasi kira kira ±0.01, sehingga setiap kesenjangan di atas ~0.02 adalah sinyal nyata. Nilai positif yang berarti pada `engine_size` menguatkan kelengkungan yang terlihat pada plot pencar dan secara independen membenarkan dijalankannya Tahap 2. Kesenjangan yang mendekati nol akan menyiratkan bahwa spesifikasi linear sudah memadai.*

## 12. Tahap 1: Keputusan Pemilihan Fitur

**Aturan.** Buang sebuah kandidat prediktor jika (a) korelasi marginalnya dengan `comb_l_100km` lemah (|ρ| di bawah sekitar 0.20 baik pada Pearson maupun Spearman), atau (b) hampir kolinier dengan prediktor lain yang dipertahankan (|ρ| di atas sekitar 0.85 dengan prediktor yang memiliki keterkaitan lebih kuat terhadap respon).

In [ ]:
candidates = ['engine_size', 'cylinders', 'year']
rows = []
for p in candidates:
    pear_y = pearson.loc[p, target]
    spear_y = spearman.loc[p, target]
    others = [c for c in candidates if c != p]
    max_partner = max(others, key=lambda c: abs(pearson.loc[p, c]))
    max_val = pearson.loc[p, max_partner]
    weak = abs(pear_y) < 0.20 and abs(spear_y) < 0.20
    collinear = abs(max_val) > 0.85
    if weak:
        decision = 'BUANG: korelasi marginal lemah dengan y'
    elif collinear:
        peer_strength = abs(pearson.loc[max_partner, target])
        if peer_strength > abs(pear_y):
            decision = f'BUANG: kolinier dengan {max_partner} (r={max_val:.2f}), peer lebih kuat terkait dengan y'
        else:
            decision = f'PERTAHANKAN: kolinier dengan {max_partner} namun lebih kuat terkait dengan y'
    else:
        decision = 'PERTAHANKAN'
    rows.append({
        'predictor': p,
        'pearson_vs_y': round(pear_y, 3),
        'spearman_vs_y': round(spear_y, 3),
        'strongest_peer': max_partner,
        'r_with_peer': round(max_val, 3),
        'decision': decision,
    })
selection = pd.DataFrame(rows)
selection

*Tabel ini mengkodekan aturan Tahap 1 yang diterapkan secara otomatis: buang jika |r| dengan y di bawah 0.20 baik pada Pearson maupun Spearman, buang jika |r| dengan peer melebihi 0.85 dan peer tersebut memiliki keterkaitan lebih kuat terhadap y, dan sisanya dipertahankan. Untuk dataset ini, aturan tersebut hanya menyisakan `engine_size`. Variabel `cylinders` dibuang karena kolinieritas dengan `engine_size`, sedangkan `year` dibuang karena korelasi marginal yang lemah. Kolom `decision` mendokumentasikan alasannya agar keputusan tetap dapat diaudit di kemudian hari.*

In [ ]:
retained = selection.loc[selection['decision'].str.startswith('PERTAHANKAN'), 'predictor'].tolist()
print('Prediktor yang dipertahankan:', retained)
print('Respon:', target)

*Apa pun yang muncul di `retained` adalah persis himpunan kolom yang akan dikonsumsi oleh notebook regresi. Karena hanya `engine_size` yang tersisa, analisis menyusut menjadi regresi polinomial dengan satu prediktor, dan matriks Z untuk prediktor linear tambahan pada M₁, M₂, dan M₃ menjadi kosong. Dalam kasus tersebut, model menyederhana menjadi y = β₀ + β₁x₁ + ε, y = β₀ + β₁x₁ + β₂x₁² + ε, dan y = β₀ + β₁x₁ + β₂x₁² + β₃x₁³ + ε.*

## 13. Lampiran: Pair Plot Lengkap

In [ ]:
sample = df.sample(n=min(3000, len(df)), random_state=RNG)
sns.pairplot(sample, plot_kws={'alpha': 0.25, 's': 8}, diag_kind='hist', height=2.0)
plt.suptitle('Pair plot (random sample of 3,000 rows)', y=1.02)
plt.show()

*Diagonal mengulang histogram per variabel; off-diagonal mengulang plot pencar bivariat dalam satu grid. Panel yang patut diperiksa ulang di sini adalah `engine_size` lawan `cylinders`. Titik titik membentuk garis yang nyaris deterministik, di mana setiap jumlah silinder memetakan ke pita ukuran mesin yang sempit. Pola tersebut adalah tanda visual dari koefisien Pearson 0.91 dan alasan mengapa `cylinders` dibuang dari model.*